In [ ]:
# ============================================================
# GOLD FACTOR RESEARCH
# NOTEBOOK 03 - INFORMATION DIFFUSION ANALYSIS
# ============================================================

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.tsa.stattools import grangercausalitytests

plt.style.use("ggplot")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

# ============================================================
# LOAD DATA
# ============================================================

df = pd.read_csv(
    "../data/processed/factor_data.csv",
    index_col=0,
    parse_dates=True
)

print("Dataset Loaded")
print(df.shape)

df.head()

# ============================================================
# VERIFY REQUIRED COLUMNS
# ============================================================

required_columns = [
    "gold",
    "dxy",
    "spy",
    "vix",
    "silver",
    "us10y",
    "future_return_5m"
]

for col in required_columns:

    if col in df.columns:

        print(f"✓ {col}")

    else:

        print(f"✗ Missing: {col}")

# ============================================================
# CREATE INFORMATION DIFFUSION FACTORS
# ============================================================

# DXY FACTORS

df["dxy_5d"] = (
    df["dxy"]
    .pct_change(5)
)

df["dxy_20d"] = (
    df["dxy"]
    .pct_change(20)
)

df["dxy_60d"] = (
    df["dxy"]
    .pct_change(60)
)

# ============================================================
# TREASURY FACTORS
# ============================================================

df["us10y_5d"] = (
    df["us10y"]
    .pct_change(5)
)

df["us10y_20d"] = (
    df["us10y"]
    .pct_change(20)
)

df["us10y_60d"] = (
    df["us10y"]
    .pct_change(60)
)

# ============================================================
# VIX FACTORS
# ============================================================

df["vix_5d"] = (
    df["vix"]
    .pct_change(5)
)

df["vix_20d"] = (
    df["vix"]
    .pct_change(20)
)

# ============================================================
# GOLD RETURNS
# ============================================================

df["gold_return"] = (
    df["gold"]
    .pct_change()
)

# ============================================================
# DXY CORRELATION ANALYSIS
# ============================================================

dxy_corr = df[
    [
        "dxy_5d",
        "dxy_20d",
        "dxy_60d",
        "future_return_5m"
    ]
].corr()

print("\nDXY Correlations")

display(
    dxy_corr[
        ["future_return_5m"]
    ]
)

# ============================================================
# TREASURY CORRELATION ANALYSIS
# ============================================================

treasury_corr = df[
    [
        "us10y_5d",
        "us10y_20d",
        "us10y_60d",
        "future_return_5m"
    ]
].corr()

print("\nTreasury Correlations")

display(
    treasury_corr[
        ["future_return_5m"]
    ]
)

# ============================================================
# VIX CORRELATION ANALYSIS
# ============================================================

vix_corr = df[
    [
        "vix_5d",
        "vix_20d",
        "future_return_5m"
    ]
].corr()

print("\nVIX Correlations")

display(
    vix_corr[
        ["future_return_5m"]
    ]
)

# ============================================================
# CROSS ASSET CORRELATION MATRIX
# ============================================================

cross_assets = [

    "gold",

    "dxy",

    "spy",

    "vix",

    "silver",

    "us10y"

]

plt.figure(
    figsize=(10,8)
)

sns.heatmap(
    df[cross_assets].corr(),
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title(
    "Cross Asset Relationship Matrix"
)

plt.tight_layout()

plt.savefig(
    "../images/cross_asset_heatmap.png"
)

plt.show()

# ============================================================
# DXY VS GOLD
# ============================================================

plt.figure(
    figsize=(12,6)
)

plt.plot(
    df["gold"] /
    df["gold"].iloc[0],
    label="Gold"
)

plt.plot(
    df["dxy"] /
    df["dxy"].iloc[0],
    label="DXY"
)

plt.legend()

plt.title(
    "Gold vs DXY"
)

plt.tight_layout()

plt.savefig(
    "../images/gold_vs_dxy.png"
)

plt.show()

# ============================================================
# TREASURY VS GOLD
# ============================================================

plt.figure(
    figsize=(12,6)
)

plt.plot(
    df["gold"] /
    df["gold"].iloc[0],
    label="Gold"
)

plt.plot(
    df["us10y"] /
    df["us10y"].iloc[0],
    label="10Y Treasury"
)

plt.legend()

plt.title(
    "Gold vs Treasury Yields"
)

plt.tight_layout()

plt.savefig(
    "../images/gold_vs_treasury.png"
)

plt.show()

# ============================================================
# LEAD-LAG ANALYSIS : DXY -> GOLD
# ============================================================

lags = range(-30, 31)

lead_lag_corr = []

for lag in lags:

    corr = (
        df["dxy"]
        .shift(lag)
        .corr(df["gold"])
    )

    lead_lag_corr.append(corr)

lead_lag_df = pd.DataFrame({

    "Lag": lags,
    "Correlation": lead_lag_corr

})

best_lag = lead_lag_df.loc[
    lead_lag_df["Correlation"]
    .abs()
    .idxmax()
]

print("\nLead Lag Analysis")

print(
    f"Strongest Relationship At Lag: {best_lag['Lag']}"
)

print(
    f"Correlation: {best_lag['Correlation']:.4f}"
)

plt.figure(
    figsize=(12,6)
)

plt.plot(
    lead_lag_df["Lag"],
    lead_lag_df["Correlation"]
)

plt.axvline(
    best_lag["Lag"],
    color="red"
)

plt.title(
    "DXY Lead-Lag Analysis"
)

plt.xlabel("Lag (Days)")

plt.ylabel("Correlation")

plt.tight_layout()

plt.savefig(
    "../images/lead_lag_analysis.png"
)

plt.show()

# ============================================================
# INFORMATION HALF-LIFE
# ============================================================

corr_values = []

for lag in range(1,31):

    value = (
        df["dxy"]
        .shift(lag)
        .corr(
            df["future_return_5m"]
        )
    )

    corr_values.append(abs(value))

corr_decay = pd.DataFrame({

    "Lag": range(1,31),
    "Correlation": corr_values

})

plt.figure(
    figsize=(10,5)
)

plt.plot(
    corr_decay["Lag"],
    corr_decay["Correlation"]
)

plt.title(
    "Information Decay Curve"
)

plt.xlabel(
    "Lag (Days)"
)

plt.ylabel(
    "Absolute Correlation"
)

plt.tight_layout()

plt.savefig(
    "../images/information_decay.png"
)

plt.show()

half_corr = (
    corr_decay["Correlation"]
    .max()
    * 0.5
)

half_life = corr_decay[
    corr_decay["Correlation"]
    <= half_corr
]

if len(half_life) > 0:

    estimated_half_life = (
        half_life.iloc[0]["Lag"]
    )

    print(
        f"\nEstimated Information Half Life: {estimated_half_life} Days"
    )

# ============================================================
# GRANGER CAUSALITY TEST
# ============================================================

print("\nRunning Granger Causality Tests")

granger_data = df[
    [
        "gold_return",
        "dxy_20d"
    ]
].dropna()

try:

    granger_results = (
        grangercausalitytests(
            granger_data,
            maxlag=5,
            verbose=False
        )
    )

    p_values = []

    for lag in range(1,6):

        p = (
            granger_results[lag][0]
            ["ssr_ftest"][1]
        )

        p_values.append(p)

    granger_df = pd.DataFrame({

        "Lag": range(1,6),
        "P_Value": p_values

    })

    print(
        "\nGranger Causality Results"
    )

    display(
        granger_df
    )

except:

    print(
        "Granger test could not be completed."
    )

# ============================================================
# INFORMATION DIFFUSION FACTOR RANKINGS
# ============================================================

diffusion_factors = [

    "dxy_5d",

    "dxy_20d",

    "dxy_60d",

    "us10y_5d",

    "us10y_20d",

    "us10y_60d",

    "vix_5d",

    "vix_20d"

]

ranking = pd.DataFrame({

    "Factor": diffusion_factors,

    "Correlation": [

        df[f].corr(
            df["future_return_5m"]
        )

        for f in diffusion_factors

    ]

})

ranking = ranking.sort_values(
    by="Correlation",
    ascending=False
)

print(
    "\nInformation Diffusion Rankings"
)

display(
    ranking
)

# ============================================================
# FACTOR RANKING CHART
# ============================================================

plt.figure(
    figsize=(10,6)
)

plt.barh(
    ranking["Factor"],
    ranking["Correlation"]
)

plt.title(
    "Information Diffusion Factors"
)

plt.tight_layout()

plt.savefig(
    "../images/information_diffusion_factors.png"
)

plt.show()

# ============================================================
# SAVE ENRICHED DATASET
# ============================================================

df.to_csv(
    "../data/processed/diffusion_data.csv"
)

print(
    "\ndiffusion_data.csv saved successfully."
)

# ============================================================
# RESEARCH SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("INFORMATION DIFFUSION RESEARCH SUMMARY")
print("=" * 60)

print("""

Research Objective

Determine whether information from
other financial markets propagates
into future Gold returns.

Markets Studied

✓ U.S. Dollar Index (DXY)

✓ Treasury Yields

✓ VIX

✓ Silver

✓ Equities

Research Methods

✓ Cross Asset Correlations

✓ Lead-Lag Analysis

✓ Information Half Life

✓ Granger Causality

✓ Factor Rankings

Deliverables

✓ Cross Asset Heatmap

✓ DXY Analysis

✓ Treasury Analysis

✓ Lead-Lag Research

✓ Information Decay Research

✓ Granger Causality Testing

✓ Diffusion Rankings

Next Notebook

04_model_and_backtest.ipynb

""")